In [34]:
import pandas as pd
from plotnine import ggplot, aes, geom_col, position_dodge, labs, theme_minimal, scale_y_continuous

In [35]:
# 1. Load Audris's updated dataset containing relationship_stable
df = pd.read_csv('../data/processed/hcmst_analysis_base_relationship_stability.csv')

In [36]:
# 2. Inspect columns to verify all variables are present
print(df.columns.tolist())

['caseid_new', 'weight1', 'weight2', 'weight3', 'ppincimp', 'ppeduc', 'q10', 'q23', 'income_ordinal', 'income_tertile', 'respondent_edu_group', 'partner_edu_group', 'respondent_edu_score', 'partner_edu_score', 'education_comparison', 'w2_q5', 'w2_q9', 'w3_q5', 'w3_q9', 'how_long_relationship', 'ppethm', 'pp2_ppethm', 'pp3_ppethm', 'q6a', 'q6b', 'relationship_stable', 'relationship_duration_years']


In [37]:
# Combine Wave 1, 2, and 3 race responses into one lowercase variable
df['respondent_race_raw'] = df['ppethm'].fillna(df['pp2_ppethm']).fillna(df['pp3_ppethm']).str.lower()

In [38]:
# Define Han's grouping dictionary
race_map = {
    'white, non-hispanic': 'White',
    'hispanic': 'Hispanic',
    'black, non-hispanic': 'Black',
    '2+ races, non-hispanic': 'Other/Multiracial',
    'other, non-hispanic': 'Other/Multiracial'
}

In [39]:
# Apply the mapping to create the respondent_race_group column
df['respondent_race_group'] = df['respondent_race_raw'].map(race_map)

Filter Data & Calculate the Mean Stability Rate

In [40]:
# 1. Remove rows with missing data in any of the 3 key variables
plot_data = df.dropna(subset=['income_tertile', 'respondent_race_group', 'relationship_stable'])

In [41]:
# 2. Group by SES Tertile and Race, then calculate the mean stability rate
summary_df = plot_data.groupby(['income_tertile', 'respondent_race_group'], as_index=False)['relationship_stable'].mean()

In [42]:
# 3. Order the SES tertiles logically from Low -> Middle -> High
summary_df['income_tertile'] = pd.Categorical(
    summary_df['income_tertile'], 
    categories=['Low', 'Middle', 'High'], 
    ordered=True
)

Build and Export the Grouped Bar Chart

In [43]:
# 1. Build the grouped bar chart using plotnine
interaction_chart = (
    ggplot(summary_df) +
    geom_col(
        aes(x='income_tertile', y='relationship_stable', fill='respondent_race_group'), 
        position=position_dodge()
    ) +
    scale_y_continuous(labels=lambda l: ["{:.0f}%".format(v * 100) for v in l]) +
    labs(
        x="Household Income Tertile", 
        y="Relationship Stability Rate (%)", 
        fill="Respondent Race", 
        title="Relationship Stability by SES and Race"
    ) +
    theme_minimal()
)

In [ ]:
# 2. Render the chart in notebook
print(interaction_chart)

<ggplot: (640 x 480)>


In [ ]:
# 3. Save the image into output/figures/ for presentation slide
interaction_chart.save('../output/figures/ses_race_stability_chart.png', width=10, height=6, dpi=300)

/Users/yuxin/Desktop/hs4002-dalmation-2026/.venv/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 10 x 6 in image.
/Users/yuxin/Desktop/hs4002-dalmation-2026/.venv/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: ../output/figures/ses_race_stability_chart.png
